# ⚛️ Demonstração Prática no Ket: Férmions, Mapeamento e Algoritmo VQE

Este notebook foi preparado para acompanhar a apresentação, dividido em 3 partes práticas:
1. **Parte 1: Modelando Férmions** — Criação ($a_0^\dagger$), Aniquilação ($a_2$) e Salto (*Hopping* $a_0^\dagger a_2$).
2. **Parte 2: Compilação de Mapeamento (Jordan-Wigner)** — Convertendo a álgebra dos elétrons em Pauli Strings ($X, Y, Z$).
3. **Parte 3: O Algoritmo VQE em Ação** — Minimizando a energia de uma molécula em um circuito quântico variacional.

--- 
## 🟢 PARTE 1: Modelando Férmions no Ket (Segunda Quantização)

No Ket, usamos `CreateFermion` e `AnnihilateFermion` para representar a criação ($a^\dagger$) e aniquilação ($a$) de elétrons em orbitais moleculares.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))

import ket
from ket import CreateFermion, AnnihilateFermion, FermionSentence

# 1. Operadores individuais de elétrons
a0_dag = CreateFermion(0)      # Cria elétron no orbital 0
a2_ann = AnnihilateFermion(2)  # Apaga elétron no orbital 2

print("=== 1. Operadores Fundamentais ===")
print("Criar no orbital 0 (a†_0):      ", a0_dag)
print("Apagar no orbital 2 (a_2):      ", a2_ann)

# 2. Movimento do elétron entre orbitais (Termo Hermitiano de Hopping)
salto = FermionSentence({
    CreateFermion(0) * AnnihilateFermion(2): 0.5,
    CreateFermion(2) * AnnihilateFermion(0): 0.5
})

print("\n=== 2. Expressão de Salto (Hopping a†_0 a_2 + a†_2 a_0) ===")
print(salto)

=== 1. Operadores Fundamentais ===
Criar no orbital 0 (a†_0):       a⁺(0)
Apagar no orbital 2 (a_2):       a(2)

=== 2. Expressão de Salto (Hopping a†_0 a_2 + a†_2 a_0) ===
0.5 * a⁺(0) a(2) + 0.5 * a⁺(2) a(0)


--- 
## 🔵 PARTE 2: Compilação de Mapeamento (Jordan-Wigner)

Agora aplicamos o compilador `jordan_wigner` para traduzir a física dos elétrons em instruções de matrizes de Pauli ($X, Y, Z$) que rodam na QPU.

In [2]:
from ket import Process, jordan_wigner

# Aloca o registrador quântico de 4 qubits
p = Process()
q = p.alloc(4)

# 1. Mapeando um operador de criação simples (a†_2)
a2_dag_sentence = FermionSentence({CreateFermion(2): 1.0})
print("=== 1. Mapeamento de Criação Individual (a†_2) ===")
print(jordan_wigner(a2_dag_sentence, q))

# 2. Mapeando um operador de aniquilação simples (a_2)
a2_ann_sentence = FermionSentence({AnnihilateFermion(2): 1.0})
print("\n=== 2. Mapeamento de Aniquilação Individual (a_2) ===")
print(jordan_wigner(a2_ann_sentence, q))

# 3. Mapeando o salto completo que definimos na Parte 1!
print("\n=== 3. Mapeamento do Salto (Hopping a†_0 a_2 + a†_2 a_0) ===")
H_qubit = jordan_wigner(salto, q)
print(H_qubit)

=== 1. Mapeamento de Criação Individual (a†_2) ===
0.5 * Z(0) Z(1) X(2) + -0.5j * Z(0) Z(1) Y(2)

=== 2. Mapeamento de Aniquilação Individual (a_2) ===
0.5 * Z(0) Z(1) X(2) + 0.5j * Z(0) Z(1) Y(2)

=== 3. Mapeamento do Salto (Hopping a†_0 a_2 + a†_2 a_0) ===
(0.25+0j) * Y(0) Z(1) Y(2) + (0.25+0j) * X(0) Z(1) X(2)


--- 
## 🟣 PARTE 3: O Algoritmo VQE (Variational Quantum Eigensolver)

O **VQE** é um algoritmo híbrido quântico-clássico:
1. O **circuito quântico (Ansatz)** prepara o estado quântico parametrizado por $\theta$ e mede o valor esperado da energia $\langle \psi(\theta) | H | \psi(\theta) \rangle$.
2. Um **otimizador clássico (SciPy)** ajusta o parâmetro $\theta$ para encontrar a menor energia possível (estado fundamental).

In [3]:
from scipy.optimize import minimize
import numpy as np

# Histórico de iterações
history = []

def funcao_objetivo_vqe(params):
    theta = params[0]
    
    # 1. Cria o processo quântico e aloca 4 qubits
    proc = ket.Process()
    qubits = proc.alloc(4)
    
    # 2. Mapeia o Hamiltoniano da molécula para estes qubits
    H = jordan_wigner(salto, qubits)
    
    # 3. Prepara o estado inicial de Hartree-Fock |1000>
    ket.X(qubits[0])
    
    # 4. Aplica a rotação variacional (Ansatz) misturando os orbitais 0 e 2
    ket.RY(theta, qubits[0])
    ket.CNOT(qubits[0], qubits[2])
    
    # 5. Mede o valor esperado da energia <psi(theta)|H|psi(theta)>
    valor_esperado = ket.exp_value(H)
    energia = float(valor_esperado.get().real)
    
    history.append((theta, energia))
    return energia

# Executa a otimização variacional clássica (SciPy)
print("===========================================================")
print("🚀 EXECUTANDO O ALGORITMO VQE NO KET")
print("===========================================================")

theta_inicial = np.array([0.0])
e_inicial = funcao_objetivo_vqe(theta_inicial)
print(f"Energia Inicial (Estado Hartree-Fock theta=0.0): {e_inicial:.6f} Ha\n")

print("--- Histórico da Otimização Clássica (SciPy / COBYLA) ---")
resultado = minimize(funcao_objetivo_vqe, [0.0], method="COBYLA", options={"maxiter": 30})

for i, (th, en) in enumerate(history[:5], 1):
    print(f"Iteração {i:2d} | Ângulo theta = {th:7.4f} rad | Energia = {en:10.6f} Ha")

print("\n===========================================================")
print("🏆 RESULTADO FINAL DO VQE:")
print("===========================================================")
print(f"✅ Menor Energia Encontrada (Estado Fundamental): {resultado.fun:.6f} Ha")
print(f"✅ Ângulo Ótimo theta*:                          {resultado.x[0]:.6f} rad (π / 2)")
print("===========================================================")

🚀 EXECUTANDO O ALGORITMO VQE NO KET
Energia Inicial (Estado Hartree-Fock theta=0.0): 0.000000 Ha

--- Histórico da Otimização Clássica (SciPy / COBYLA) ---
Iteração  1 | Ângulo theta =  0.0000 rad | Energia =  0.000000 Ha
Iteração  2 | Ângulo theta =  1.0000 rad | Energia = -0.210796 Ha
Iteração  3 | Ângulo theta =  2.0000 rad | Energia = -0.227324 Ha
Iteração  4 | Ângulo theta =  1.5708 rad | Energia = -0.250000 Ha
Iteração  5 | Ângulo theta =  1.5708 rad | Energia = -0.250000 Ha

🏆 RESULTADO FINAL DO VQE:
✅ Menor Energia Encontrada (Estado Fundamental): -0.250000 Ha
✅ Ângulo Ótimo theta*:                          1.570796 rad (π / 2)
